In [1]:
import numpy as np
from collections import deque  # Import deque for efficient queue operations

In [2]:
class Agent:
    def __init__(self):
        # self.world = [
        #     ["S", "", "B", "P"],
        #     ["W", "B, S, G", "P", "B"],
        #     ["S", "", "B", ""],
        #     ["", "B", "P", "B"],
        # ]
        # self.world = [
        #     ["", "B", "P", "B, S, G"],
        #     ["B", "P", "B, S", "W"],
        #     ["", "B", "B", "S"],
        #     ["S", "B", "P", "B"],
        # ]
        self.world = [
            ["", "B", "P", "B, S, G"],
            ["", "", "B, S", "W"],
            ["", "", "B", "S"],
            ["S", "B", "P", "B"],
        ]
        self.alive = True
        self._position = (1, 1)  # (row, col) in board coordinates, row 1 = bottom
        self.costs = 0
        self._finish = False
        self.has_gold = False
        self.has_arrow = True

    def _coords_to_index(self, pos):
        row, col = pos
        return len(self.world) - row, col - 1

    def _adjacent_cells(self, pos=None):
        if pos is None:
            pos = self._position
        adj = []
        for dr, dc in [[0, 1], [0, -1], [-1, 0], [1, 0]]:
            r, c = pos[0] + dr, pos[1] + dc
            if 1 <= r <= 4 and 1 <= c <= 4:
                adj.append((r, c))  # tuples, so cells can be dict keys / set items
        return adj

    def _perceive(self):
        if not self.alive:
            print(f"Cannot perceive. DEAD at {self._position}")
            return [None, None, None]

        breeze, stench = False, False
        for r, c in self._adjacent_cells():
            i, j = self._coords_to_index([r, c])
            cell = self.world[i][j]
            if "P" in cell:
                breeze = True
            if "W" in cell:
                stench = True
        # Glitter is perceived only in the cell that contains the gold
        i, j = self._coords_to_index(self._position)
        gold = "G" in self.world[i][j]
        return [breeze, stench, gold]

    def _check_hazards(self):
        row, col = self._coords_to_index(self._position)
        cell = self.world[row][col]
        if "P" in cell or "W" in cell:
            self.alive = False
            self.costs -= 1000
            print(f"Agent encountered hazard at {self._position}. DEAD.")
        if "G" in cell:
            print(f"Agent found the treasure at {self._position}!")
            self._finish = True
        return self.alive

    def is_valid(self, move):
        r, c = self._position[0] + move[0], self._position[1] + move[1]
        if not (1 <= r <= len(self.world) and 1 <= c <= len(self.world[0])):
            return False
        if not self.alive:
            print("Agent is dead")
            return False
        return True

    def move_agent(self, move):
        valid = self.is_valid(move)
        if valid:
            self._position = self._position[0] + move[0], self._position[1] + move[1]
            self.costs -= 1
            self._check_hazards()

    def grab(self):
        i, j = self._coords_to_index(self._position)
        if "G" in self.world[i][j]:
            self.has_gold = True
            print(f"Agent grabbed the gold at {self._position}")

    def climb(self):
        if self._position == (1, 1) and self.has_gold:
            self.costs += 1000
            print("Agent climbed out with the gold!")

    def shoot(self, direction):
        """Shoot the arrow in a straight line (dr, dc). Returns True if the Wumpus screams."""
        if not self.alive or not self.has_arrow:
            return False
        self.has_arrow = False
        self.costs -= 10
        r, c = self._position[0] + direction[0], self._position[1] + direction[1]
        # The arrow flies until it hits the Wumpus or a wall
        while 1 <= r <= len(self.world) and 1 <= c <= len(self.world[0]):
            i, j = self._coords_to_index((r, c))
            if "W" in self.world[i][j]:
                self.world[i][j] = self.world[i][j].replace("W", "")  # Wumpus is dead
                print(f"Arrow hit the Wumpus at {(r, c)}. SCREAM!")
                return True
            r, c = r + direction[0], c + direction[1]
        print("Arrow missed.")
        return False

In [3]:
# Define the BFS function
def bfs(agent, start, is_goal, allowed):
    parent = {start: None}  # Dictionary to keep track of parent nodes
    queue = deque([start])  # Initialize the queue with the starting node
    while queue:  # While there are still nodes to process
        node = queue.popleft()  # Dequeue a node from the front of the queue
        if is_goal(node):  # Check if the current node is the goal
            path = []  # Initialize an empty path
            while node != start:  # Backtrack to construct the path
                path.append(node)  # Add the current node to the path
                node = parent[node]  # Move to the parent node
            return path[::-1]  # Return the reversed path (from start to goal)
        for nb in agent._adjacent_cells(node):  # Iterate through adjacent cells
            if (
                nb in allowed and nb not in parent
            ):  # If the neighbor hasn't been visited and is allowed
                parent[nb] = node  # Set the current node as the parent of the neighbor
                queue.append(nb)  # Enqueue the neighbor for processing
    return None  # Return None if no path to the goal was found

In [4]:
class KnowledgeBase:
    """What the agent has learned from its percepts. It never reads agent.world."""

    def __init__(self):
        self.cells = {(r, c) for r in range(1, 5) for c in range(1, 5)}
        self.visited = set()
        self.no_pit = set()  # cells known to have no pit
        self.no_wumpus = set()  # cells known to have no Wumpus
        self.wumpus_candidates = set(
            self.cells
        )  # cells where the Wumpus could still be

    def tell(self, pos, neighbors, breeze, stench):
        # The agent is alive here, so this cell is safe
        self.visited.add(pos)
        self.no_pit.add(pos)
        self.no_wumpus.add(pos)

        # No breeze -> no neighbor is a pit; no stench -> no neighbor is the Wumpus
        if not breeze:
            self.no_pit.update(neighbors)
        if not stench:
            self.no_wumpus.update(neighbors)
        else:
            # The Wumpus must be adjacent to every cell where a stench is perceived
            self.wumpus_candidates &= set(neighbors)
        self._update_wumpus()

    def tell_shot(self, line, scream):
        if scream:
            self.wumpus_candidates = set()  # the Wumpus is dead
        else:
            self.wumpus_candidates -= set(line)  # the arrow flew through these cells
        self._update_wumpus()

    def _update_wumpus(self):
        # There is exactly one Wumpus: every cell that is not a candidate is Wumpus-free
        self.wumpus_candidates -= self.no_wumpus
        self.no_wumpus |= self.cells - self.wumpus_candidates
        # A cell never holds both a pit and the Wumpus: once located, its cell is pit-free
        if len(self.wumpus_candidates) == 1:
            self.no_pit |= self.wumpus_candidates

    def safe_cells(self):
        return self.no_pit & self.no_wumpus

In [5]:
def follow(agent, path):
    # Turn each target cell into a move (dr, dc) and execute it
    for cell in path:
        move = (cell[0] - agent._position[0], cell[1] - agent._position[1])
        agent.move_agent(move)
        print(f"  -> {agent._position}  costs={agent.costs}")


def line_of_fire(pos, direction):
    # Cells the arrow flies through, from pos to the wall
    line = []
    r, c = pos[0] + direction[0], pos[1] + direction[1]
    while 1 <= r <= 4 and 1 <= c <= 4:
        line.append((r, c))
        r, c = r + direction[0], c + direction[1]
    return line


def plan_shot(agent, kb):
    # Shooting only helps if a pit-free, unvisited cell is blocked by the Wumpus alone
    targets = (kb.no_pit & kb.wumpus_candidates) - kb.visited
    safe = kb.safe_cells()
    aim = {}  # firing spot -> direction (dr, dc) towards a target
    for t in targets:
        for s in safe:
            if s != t and (s[0] == t[0] or s[1] == t[1]):
                aim[s] = ((t[0] > s[0]) - (t[0] < s[0]), (t[1] > s[1]) - (t[1] < s[1]))
    path = bfs(agent, agent._position, lambda c: c in aim, safe)
    if path is None:
        return None
    spot = path[-1] if path else agent._position
    return path, aim[spot]


def run(agent):
    kb = KnowledgeBase()

    # 1. Explore: perceive, update the KB, go to the nearest safe unvisited cell
    while agent.alive:
        pos = agent._position
        breeze, stench, gold = agent._perceive()
        kb.tell(pos, agent._adjacent_cells(pos), breeze, stench)
        print(f"At {pos}: breeze={breeze}, stench={stench}, gold={gold}")

        if gold:
            agent.grab()
            break

        safe = kb.safe_cells()
        path = bfs(agent, pos, lambda c: c in safe and c not in kb.visited, safe)
        if path is not None:
            follow(agent, path)
            continue

        # No safe cell left: try to clear the way with the arrow
        plan = plan_shot(agent, kb) if agent.has_arrow else None
        if plan is None:
            print("No safe cell left to explore.")
            break
        path, direction = plan
        follow(agent, path)
        scream = agent.shoot(direction)
        kb.tell_shot(line_of_fire(agent._position, direction), scream)
        print(f"  shot {direction} from {agent._position}  costs={agent.costs}")

    # 2. Go back to (1, 1) through visited (safe) cells and climb out
    if agent.alive:
        follow(agent, bfs(agent, agent._position, lambda c: c == (1, 1), kb.visited))
        agent.climb()

    print(f"Final score: {agent.costs}")


run(Agent())

At (1, 1): breeze=False, stench=False, gold=False
  -> (1, 2)  costs=-1
At (1, 2): breeze=True, stench=False, gold=False
  -> (1, 1)  costs=-2
  -> (2, 1)  costs=-3
At (2, 1): breeze=False, stench=False, gold=False
  -> (2, 2)  costs=-4
At (2, 2): breeze=False, stench=False, gold=False
  -> (2, 3)  costs=-5
At (2, 3): breeze=True, stench=False, gold=False
  -> (2, 2)  costs=-6
  -> (3, 2)  costs=-7
At (3, 2): breeze=False, stench=False, gold=False
  -> (3, 3)  costs=-8
At (3, 3): breeze=True, stench=True, gold=False
  -> (3, 2)  costs=-9
  -> (3, 1)  costs=-10
At (3, 1): breeze=False, stench=False, gold=False
  -> (4, 1)  costs=-11
At (4, 1): breeze=False, stench=False, gold=False
  -> (4, 2)  costs=-12
At (4, 2): breeze=True, stench=False, gold=False
  -> (3, 2)  costs=-13
Arrow hit the Wumpus at (3, 4). SCREAM!
  shot (0, 1) from (3, 2)  costs=-23
At (3, 2): breeze=False, stench=False, gold=False
  -> (3, 3)  costs=-24
  -> (3, 4)  costs=-25
At (3, 4): breeze=False, stench=False, gol